In [19]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

from gnss_pre.dataset_gen import load_dataset_from_paths
from utils.data_io import get_imu_data_from_path, get_gnss_data_from_path

ratio = 100


In [20]:
from sklearn.preprocessing import StandardScaler

def get_dataset_from_single_path(file_path, step):
    # step指gnss频率，这里是1s
    if step < 1:
        raise ValueError("step should be a positive integer")
    imu_data = get_imu_data_from_path(file_path, False)[ratio:][:]
    gnss_data = get_gnss_data_from_path(file_path, False)
    gnss_data = np.diff(gnss_data, axis=0)
    features = []
    labels = []
    for i in range(step - 1, len(gnss_data) - 1):
        labels.append(gnss_data[i, :])
        features.append(imu_data[(i - step + 1)*ratio:(i + 1)*ratio, :])

    # 加入归一化
    N, T, F = features.shape

    # ========== 1. 特征归一化 ==========
    scaler_x = StandardScaler()
    features_2d = features.reshape(N * T, F)   # (N*T, F)
    features_norm_2d = scaler_x.fit_transform(features_2d)
    features_norm = features_norm_2d.reshape(N, T, F)

    # ========== 2. 标签归一化 ==========
    scaler_y = StandardScaler()
    labels_norm = scaler_y.fit_transform(labels)

    features = features_norm.copy()
    labels = labels_norm.copy()


    return np.array(features, dtype=np.float32), \
           np.array(labels, dtype=np.float32)


In [21]:
import os

def get_subfolders(parent_dir):
    """
    获取指定文件夹下的所有子文件夹（不递归）

    :param parent_dir: 父文件夹路径
    :return: 子文件夹路径列表
    """
    subfolders = [os.path.join(parent_dir, f)
                  for f in os.listdir(parent_dir)
                  if os.path.isdir(os.path.join(parent_dir, f))]
    return subfolders

folders = get_subfolders("/Users/yangyu/PycharmProjects/gnss-ins-sim/sim_data_gen/sim_files/saved_file/default_slow")

time_steps = 3
from gnss_pre.dataset_gen import load_dataset_from_paths
file_path = "/Users/yangyu/PycharmProjects/gnss-ins-sim/sim_data_gen/sim_files/saved_file/default_slow/2025-11-27-20-30-57"
features, labels = load_dataset_from_paths(folders, 3)

In [25]:
from gnss_pre.DeepDav import get_dataloader

train_loader = get_dataloader(folders, 10, 64)

Dataset loaded.
Features shape: (52326, 1000, 9)
Labels shape: (52326, 6)


In [31]:
from torch import nn
from gnss_pre.DeepDav import train_model, LSTMModel, WeightedMAE

# 获取 IMU 单帧输入维度
sample_x, _ = next(iter(train_loader))
input_dim = sample_x.shape[-1]  # e.g., 6
def train_model(train_loader, input_dim, epochs=10, lr=1e-3, device="mps"):
    device = torch.device(device if torch.mps.is_available() else "cpu")
    print(f'traing model on {device}')
    model = LSTMModel(input_size=input_dim).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    signals_weights_tensor = np.array([3.8, 3.9, 7.6, 1, 1, 5.5])
    criterion = WeightedMAE(signals_weights_tensor)


    for epoch in range(epochs):
        model.train()
        total_loss = 0
        count = 0

        for x, y in train_loader:
            x = x.to(device)
            y = y.to(device)

            pred = model(x)
            loss = criterion(pred, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            count += 1

        print(f"Epoch {epoch+1}/{epochs}, Loss = {total_loss/count:.4f}")

    return model

# 训练
model = train_model(train_loader, input_dim, epochs=500, lr=1e-2)

traing model on mps
Epoch 1/500, Loss = 17.2973
Epoch 2/500, Loss = 17.2970
Epoch 3/500, Loss = 17.2975
Epoch 4/500, Loss = 17.2963
Epoch 5/500, Loss = 17.2961
Epoch 6/500, Loss = 17.2962
Epoch 7/500, Loss = 17.2970
Epoch 8/500, Loss = 17.2971
Epoch 9/500, Loss = 17.2969
Epoch 10/500, Loss = 17.2976
Epoch 11/500, Loss = 17.2969
Epoch 12/500, Loss = 17.2972


KeyboardInterrupt: 